In [10]:
import sys
import random
import math

class Game:

    def __init__(self):
        self.last_difficulty = None
        self.state = "menu"
        self.game_properties = {
            "menu": self.menu,
            "difficulty": self.difficulty,
            "easy": lambda: self.game_prompts(1, 10, math.inf),
            "medium": lambda: self.game_prompts(1, 100, 5),
            "hard": lambda: self.game_prompts(1, 1000, 3),
            "continue": self.game_end
        }

    def run(self):
        logger.info("Game started")
        while self.state:
            self.state = self.game_properties[self.state]()
            
    def menu(self):
        print("Play\nExit")
        while True:
            menu_input = input().lower()
            if menu_input in {"play"}:
                return "difficulty"
            elif menu_input in {"exit"}:
                print("Thx for playing!")
                sys.exit()
            else:
                print("Invalid command!")
    
    def difficulty(self):
        print("Easy\nMedium\nHard\nReturn")
        
        while True:
            difficulty_input = input().lower()
            if difficulty_input in {"easy"}:
                self.last_difficulty = "easy"
                return "easy"
            elif difficulty_input in {"medium"}:
                self.last_difficulty = "medium"
                return "medium"
            elif difficulty_input in {"hard"}:
                self.last_difficulty = "hard"
                return "hard"
            elif difficulty_input in {"return"}:
                return "menu"
            else:
                print("Invalid choice!")
                
    def game_prompts(self, min_range, max_range, attempts_limit):
        random_number = random.randint(min_range, max_range)
        print("Guess!")
        while True:
            try:
                guess = int(input())
                    
                if (guess > random_number):
                    print ("Too high!")
                    attempts_limit -= 1
                elif (guess < random_number):
                    print ("Too low!")
                    attempts_limit -= 1
                elif (guess == random_number):
                    print(f"Great guess!\nTry again?\nYes/No")
                    return "continue"

                difference = abs(random_number - guess)

                if (25 >= difference > 15):
                    print("Cold!")
                elif (100 >= difference > 25):
                    print("Super cold!")
                elif (difference > 100):
                    print("Extremely cold!")
                elif (15 >= difference > 10):
                    print("Warm!")
                elif (10 >= difference > 5):
                    print("Very warm!")
                elif (self.last_difficulty != "easy" and 5 >= difference > 0):
                    print("Really hot!")
                        
                if (attempts_limit != math.inf and attempts_limit > 0):
                    print(f"Attempts left: {attempts_limit}")
                    continue
                elif (attempts_limit == 0):
                    print(f"No attempts left!\nThe number was {random_number}\nTry again?\nYes/No")
                    return "continue"
                        
            except ValueError:
                print("Invalid guess!")
                    
    def game_end(self):
        while True:
            choice = input().lower()
            if (choice == "yes"):
                return self.last_difficulty
            elif (choice == "no"):
                return "menu"
            else:
                print("Invalid command!")
                print("Try again?\nYes/No")
                    
game = Game()
game.run()

Play
Exit


 play


Easy
Medium
Hard
Return


 easy


Guess!


...

 return


Play
Exit


 exit


Thx for playing!


SystemExit: 

In [ ]:
'''
game logic is moved again to a single function isntead of seperating into different methods in order to eliminate repetitive logic
added tests for the menu - play(continue to difficulty selection), and exit(sys.exit), difficulty, for the guessing check,  
for the return to menu from difficulty selection check, for the attempts check, for the continuing the same difficulty check 
and for the end the current game difficulty check
'''
#read logs to create stats

In [178]:
import unittest
from unittest.mock import patch

class TestGame(unittest.TestCase):

    @patch("builtins.input", side_effect = ["asd", "213","play"])
    def test_menu_play(self, mock_input):
        game = Game()
        result = game.menu()
        self.assertEqual(result, "difficulty")

    def test_menu_exit(self):
        game = Game()
        with patch("builtins.input", side_effect = ["sdsa", "321", "exit"]):
            with self.assertRaises(SystemExit):
                game.menu()

    @patch("builtins.input", side_effect = ["asd", "123", "return"])
    def test_return_invalid_then_correct(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "menu")
    
    @patch("builtins.input", return_value = "easy")
    def test_difficulty_easy(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "easy")
        self.assertEqual(game.last_difficulty, "easy")

    @patch("builtins.input", side_effect = ["wrong", "medium"])
    def test_invalid_then_medium(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "medium")
        self.assertEqual(game.last_difficulty, "medium")

    @patch("random.randint", return_value = 7)
    @patch("builtins.input", side_effect = ["2", "3", "9", "7"])
    def test_game_prompts_win_on_correct_guess(self, mock_input, mock_randint):
        game = Game()
        result = game.game_prompts(1, 10, math.inf)
        self.assertEqual(result, "continue")

    @patch("builtins.input", side_effect = ["asdas", "wrong", "hard"])
    def test_invalid_difficulty_then_hard(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "hard")
        self.assertEqual(game.last_difficulty, "hard")

    @patch("random.randint", return_value = 44)
    @patch("builtins.input", side_effect = ["2", "45", "60", "30", "66"])
    def test_exceeding_attempt_limits(self, mock_input, mock_randint):
        game = Game()
        result = game.game_prompts(1, 100, 5)
        self.assertEqual(result, "continue")

    @patch("builtins.input", side_effect = ["sa", "123", "no"])
    def test_invalid_game_end_input_return(self, mock_input):
        game = Game()
        result = game.game_end()
        self.assertEqual(result, "menu")

    @patch("builtins.input", side_effect = ["sda", "213", "yes"])
    def test_invalid_game_end_input_continue_last_difficulty(self, mock_input):
        game = Game()
        game.last_difficulty = "hard"
        result = game.game_end()
        self.assertEqual(result, "hard")

In [180]:
#@patch - switches given part in the code
#"builtins.input" - switches input in the code
#"random.randint" - switches the random_number in the code
#side_effect - used for multiple inputs
#return_value = "easy" - the input that "builtins.input" switches, used for a value that is given once
#mock_input - is just needed to be there
#result = takes the real value from the code
#self.assertEqual(result, "easy") - compares result to the given string - "easy" if it doesnt match an error is given
#self.assertEqual(game.last_difficulty, "easy") - checks the the code's memory variable

In [182]:
suite = unittest.TestLoader().loadTestsFromTestCase(TestGame)
unittest.TextTestRunner(verbosity = 2).run(suite)

test_difficulty_easy (__main__.TestGame.test_difficulty_easy) ... ok
test_exceeding_attempt_limits (__main__.TestGame.test_exceeding_attempt_limits) ... ok
test_game_prompts_win_on_correct_guess (__main__.TestGame.test_game_prompts_win_on_correct_guess) ... ok
test_invalid_difficulty_then_hard (__main__.TestGame.test_invalid_difficulty_then_hard) ... ok
test_invalid_game_end_input_continue_last_difficulty (__main__.TestGame.test_invalid_game_end_input_continue_last_difficulty) ... ok
test_invalid_game_end_input_return (__main__.TestGame.test_invalid_game_end_input_return) ... ok
test_invalid_then_medium (__main__.TestGame.test_invalid_then_medium) ... ok
test_menu_exit (__main__.TestGame.test_menu_exit) ... ok
test_menu_play (__main__.TestGame.test_menu_play) ... ok
test_return_invalid_then_correct (__main__.TestGame.test_return_invalid_then_correct) ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.009s

OK


Easy
Medium
Hard
Return
Guess!
Too low!
Super cold!
Attempts left: 4
Too high!
Really hot!
Attempts left: 3
Too high!
Cold!
Attempts left: 2
Too low!
Warm!
Attempts left: 1
Too high!
Cold!
No attempts left!
The number was 44
Try again?
Yes/No
Guess!
Too low!
Really hot!
Too low!
Really hot!
Too high!
Really hot!
Great guess!
Try again?
Yes/No
Easy
Medium
Hard
Return
Invalid choice!
Invalid choice!
Invalid command!
Try again?
Yes/No
Invalid command!
Try again?
Yes/No
Invalid command!
Try again?
Yes/No
Invalid command!
Try again?
Yes/No
Easy
Medium
Hard
Return
Invalid choice!
Play
Exit
Invalid command!
Invalid command!
Thx for playing!
Play
Exit
Invalid command!
Invalid command!
Easy
Medium
Hard
Return
Invalid choice!
Invalid choice!


'\n    @patch("builtins.input", side_effect = ["wrong", "medium"])\n    def test_invalid_then_medium(self, mock_input):\n        game = Game()\n        result = game.difficulty()\n        self.assertEqual(result, "medium")\n        self.assertEqual(game.last_difficulty, "medium")\n'